# Chuard — explicabilidad del ganador real

Explicamos XGBoost sin derivadas, calibrado con sigmoid, en la escala de su
**probabilidad final de Churn=1**. El notebook verifica el bundle propio y su referencia
exclusiva de entrenamiento. No entrena ni usa prueba para análisis o selección.

In [ ]:
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "src").is_dir() and (root.parent / "src").is_dir():
    root = root.parent
if not (root / "src").is_dir():
    raise RuntimeError("Abra el notebook desde la raíz de Chuard o notebooks/.")
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

In [ ]:
import time

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import (
    EXPLANATION_REPORT_DIR,
    MODEL_PATH,
    RANDOM_STATE,
    SHAP_ADDITIVITY_ATOL,
    SHAP_MAX_GLOBAL_ROWS,
)
from src.data import dataset_sha256
from src.explain import get_explanation_service, save_explanation_plots
from src.inference import score_customers
from src.reporting import load_frozen_resources
from src.schema import FEATURE_LABELS

resources = load_frozen_resources()
bundle = resources.bundle
frozen_hash = dataset_sha256(MODEL_PATH)
customers = resources.training.sample(
    n=min(SHAP_MAX_GLOBAL_ROWS, len(resources.training)), random_state=RANDOM_STATE
)
started = time.perf_counter()
result = get_explanation_service(bundle).explain(customers)
seconds = time.perf_counter() - started
metadata = save_explanation_plots(result, EXPLANATION_REPORT_DIR)
assert result.scale == "probability" and result.model_scope == "complete_estimator"
error = np.max(
    np.abs(result.base_values + result.values.sum(axis=1).to_numpy() - result.probabilities)
)
assert error <= SHAP_ADDITIVITY_ATOL
print("Modelo:", bundle["model_name"], "versión", bundle["model_version"])
print("Método:", result.method, "escala:", result.scale)
print(
    "Clientes de entrenamiento:",
    len(customers),
    "referencia:",
    len(bundle["explanation_context"]["background"]),
)
print("Error máximo de reconstrucción:", error)
print(f"Tiempo observado de explicación: {seconds:.2f} s; no garantiza latencia web.")

## Importancia global

Cada punto es un cliente de la muestra de entrenamiento. La contribución positiva
aumenta la probabilidad que asigna el modelo respecto al valor base; la negativa la
reduce. Los colores representan valores bajos/altos de variables numéricas; las
categorías se muestran en gris, sin imponerles un orden numérico.

El método permutation explica el estimador completo, incluida la calibración.
TreeExplainer sobre el árbol subyacente no explicaría automáticamente esa misma salida.
Los híbridos de la explicación pueden romper dependencias entre servicios: son una
aproximación interventional del modelo y no efectos causales.

In [ ]:
display(Image(filename=str(EXPLANATION_REPORT_DIR / "shap_global.png")))
importance = result.values.abs().mean().sort_values(ascending=False)
importance.index = [FEATURE_LABELS[name] for name in importance.index]
display(importance.rename("Media absoluta SHAP (probabilidad)").to_frame())
display(
    Markdown(
        "Las variables principales describen a este modelo en esta muestra; "
        "no demuestran por qué una persona abandona ni el efecto de cambiar un contrato."
    )
)

## Explicación individual y consistencia

Se elige reproduciblemente el primer cliente de la muestra, sin buscar un caso
favorable. El waterfall conserva las contribuciones restantes agrupadas. Los cinco
factores de la API tienen un remanente: no son por sí solos toda la descomposición.
En esta escala, 0,01 de contribución son 1 punto porcentual de probabilidad del modelo.

In [ ]:
display(customers.head(1))
display(Image(filename=str(EXPLANATION_REPORT_DIR / "shap_individual.png")))
factors = pd.DataFrame(result.factors(0))
display(
    factors[
        [
            "label",
            "display_value",
            "direction",
            "contribution",
            "scale",
            "base_value",
            "explained_value",
            "remaining_contribution",
        ]
    ]
)
full_sum = float(result.base_values[0] + result.values.iloc[0].sum())
print("Valor base:", result.base_values[0], "probabilidad reconstruida:", full_sum)
batch = score_customers(customers.head(3), bundle)
for position in range(len(batch)):
    individual = score_customers(customers.iloc[[position]], bundle)
    np.testing.assert_allclose(
        individual.churn_probability, batch.churn_probability.iloc[[position]], atol=1e-12, rtol=0
    )
    assert individual.top_factors.iloc[0] == batch.top_factors.iloc[position]
print("Probabilidades y factores individual/lote coinciden en tres clientes de entrenamiento.")
assert dataset_sha256(MODEL_PATH) == frozen_hash

## Entrega y límites

Se guardan `shap_global.png`, `shap_individual.png`, `shap_values.csv` y
`explanation_metadata.json` en `reports/explain/`. El proceso no modifica el modelo.
La muestra de 100 clientes no representa automáticamente todas las cargas futuras.
No se garantiza un tiempo para lotes de 10.000 filas; la web debe medir y comunicar
límites operativos antes de prometer capacidad, sin cambiar el contrato silenciosamente.

SHAP describe la predicción del modelo; no justifica discriminación por género o edad
ni demuestra que una recomendación vaya a reducir el abandono. Las recomendaciones
y su presentación quedan en el frente 3.